# 07 路 Even-split experiment
Same model/settings as 05. Only change: training images drawn evenly from sagittal / coronal / axial (same total, 4,092). Val = the same 56 sagittal knees as 05, plus per-plane and knee-level checks at the end.

In [ ]:
# Cell 1: imports, seed, labels, device
import os, numpy as np, pandas as pd, torch, timm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42)
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
print(DEVICE)

In [ ]:
# Cell 2: load all 3 planes, build an even train split (same total as baseline)
BASE_IN = '/kaggle/input/notebooks/alextanasescu/02-preprocess-test-alex'
PLANES = ['sagittal', 'coronal', 'axial']

mans = {}
for p in PLANES:
    m = pd.read_csv(f'{BASE_IN}/manifest_{p}.csv')
    m['path'] = f'{BASE_IN}/images_{p}/' + m['StudyInstanceUID'] + '.npy'
    m['plane'] = p
    mans[p] = m

train_df = pd.concat([mans[p][mans[p]['split'] == 'train'] for p in PLANES])
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)   # shuffle planes together

val_by_plane = {p: mans[p][mans[p]['split'] == 'val'].reset_index(drop=True) for p in PLANES}
val_df = val_by_plane['sagittal']   # same 56 knees/images as the baseline

print('train per plane:', train_df['plane'].value_counts().to_dict())
print('val per plane:', {p: len(v) for p, v in val_by_plane.items()})
print('first file exists:', os.path.exists(train_df['path'].iloc[0]))

In [ ]:
# Cell 3: Dataset (temporary, Cristina will take this over)
class KneeDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        vol = np.load(row['path'])                        # (24, 224, 224) uint8
        img = vol[12:13].astype(np.float32) / 255.0       # middle slice -> (1, 224, 224), 0-1
        img = (img - 0.45) / 0.225                        # centre around 0, spread around 1
        y = row[LABELS].values.astype(np.float32)         # 12 soft labels
        return torch.from_numpy(img), torch.from_numpy(y), row['StudyInstanceUID']

In [ ]:
# Cell 4: loaders + sanity checks (val = sagittal, for a fair comparison with 05)
train_loader = DataLoader(KneeDataset(train_df), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(KneeDataset(val_df), batch_size=32, shuffle=False, num_workers=2)

images, labels, ids = next(iter(train_loader))
expected = train_df.drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID').loc[list(ids), LABELS].values.astype(np.float32)
assert np.allclose(labels.numpy(), expected), 'labels do not match their IDs!'

print(images.shape, labels.shape)
print(round(float(images.mean()), 2), round(float(images.std()), 2))
print(len(train_df), len(val_df))

In [ ]:
# Cell 5: model, loss, optimizer
def make_model():
    return timm.create_model('resnet18', pretrained=True, num_classes=12, in_chans=1).to(DEVICE)

loss_fn = torch.nn.BCEWithLogitsLoss()
model = make_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M parameters')

In [ ]:
# Cell 6: overfit test on 8 images (loss should fall toward ~0.1)
model = make_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
x, y, _ = next(iter(train_loader))
x, y = x[:8].to(DEVICE), y[:8].to(DEVICE)

model.train()
for step in range(150):
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()
    if step % 25 == 0:
        print(step, round(loss.item(), 4))

In [ ]:
# Cell 7: evaluate = mean per-finding AUC on the val set (temporary until Filip's scorer)
def evaluate(model, loader):
    model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for x, y, _ in loader:
            probs.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
            ys.append(y.numpy())
    probs = np.concatenate(probs)
    ys = np.concatenate(ys) >= 0.5
    per = {}
    for i, name in enumerate(LABELS):
        if ys[:, i].min() != ys[:, i].max():          # skip findings with only one class in val
            per[name] = roc_auc_score(ys[:, i], probs[:, i])
    return float(np.mean(list(per.values()))), per

In [ ]:
# Cell 8: train, recording loss + AUC every epoch
torch.manual_seed(42); np.random.seed(42)
model = make_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

EPOCHS = 10
SAVE_PATH = '/kaggle/working/resnet18_all_planes_10ep.pt'
history, best_auc = [], 0.0

for epoch in range(EPOCHS):
    model.train()
    losses = []
    for images, labels, _ in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    train_loss = float(np.mean(losses))
    val_auc, per_finding = evaluate(model, val_loader)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_auc': val_auc})
    print(f'epoch {epoch}: train loss {train_loss:.4f} | val AUC {val_auc:.4f}')

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), SAVE_PATH)

history = pd.DataFrame(history)
history.to_csv('/kaggle/working/history.csv', index=False)
print('best val AUC:', round(best_auc, 4))

In [ ]:
# Cell 9: predictions on the val set from the best model
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

probs, ys, ids = [], [], []
with torch.no_grad():
    for images, labels, sid in val_loader:
        probs.append(torch.sigmoid(model(images.to(DEVICE))).cpu().numpy())
        ys.append(labels.numpy())
        ids += list(sid)

probs = np.concatenate(probs)            # (56, 12)
ys = np.concatenate(ys)
y_bin = (ys >= 0.5).astype(int)

pd.DataFrame(probs, columns=LABELS).assign(StudyInstanceUID=ids).to_csv('/kaggle/working/val_preds.csv', index=False)
print(probs.shape)
print('positives per finding in val:', dict(zip(LABELS, y_bin.sum(0))))

In [ ]:
# Cell 10: training curve
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(history['epoch'], history['train_loss'], 'o-', label='train loss')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss')
ax2 = ax1.twinx()
ax2.plot(history['epoch'], history['val_auc'], 's-', color='tab:orange', label='val AUC')
ax2.set_ylabel('val AUC')
fig.legend(loc='center right')
plt.title('resnet18 路 all planes, 10 ep')
plt.tight_layout(); plt.savefig('/kaggle/working/training_curve.png', dpi=120); plt.show()

In [ ]:
# Cell 11: ROC curve per finding
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, (name, ax) in enumerate(zip(LABELS, axes.flat)):
    ax.plot([0, 1], [0, 1], '--', color='grey', lw=1)          # coin-flip line
    if y_bin[:, i].min() == y_bin[:, i].max():
        ax.set_title(f'{name}\n(only one class in val)'); continue
    fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
    ax.plot(fpr, tpr, lw=2)
    ax.set_title(f'{name}  AUC {roc_auc_score(y_bin[:, i], probs[:, i]):.2f}  (n+={y_bin[:, i].sum()})')
    ax.set_xlabel('false alarms'); ax.set_ylabel('caught')
plt.tight_layout(); plt.savefig('/kaggle/working/roc_curves.png', dpi=120); plt.show()

In [ ]:
# Cell 12: confusion matrix per finding
THRESH = 0.5
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, (name, ax) in enumerate(zip(LABELS, axes.flat)):
    cm = confusion_matrix(y_bin[:, i], (probs[:, i] >= THRESH).astype(int), labels=[0, 1])
    ax.imshow(cm, cmap='Blues')
    for r in range(2):
        for c in range(2):
            ax.text(c, r, cm[r, c], ha='center', va='center', fontsize=14)
    ax.set_xticks([0, 1], ['pred no', 'pred yes']); ax.set_yticks([0, 1], ['true no', 'true yes'])
    ax.set_title(name)
plt.suptitle(f'Confusion matrices (cutoff {THRESH})')
plt.tight_layout(); plt.savefig('/kaggle/working/confusion_matrices.png', dpi=120); plt.show()

In [ ]:
# Cell 13: positive rates, train labels vs real val labels
print(pd.DataFrame({'train_soft_mean': train_df[LABELS].mean(),
                    'train_>=0.5':     train_df[LABELS].ge(0.5).mean(),
                    'val':             val_df[LABELS].mean()}).round(2))

In [ ]:
# Cell 14: evaluate on every plane + knee-level (average the 3 planes per knee)
def predict(df):
    loader = DataLoader(KneeDataset(df), batch_size=32, shuffle=False, num_workers=2)
    model.eval(); out = []
    with torch.no_grad():
        for x, _, sid in loader:
            p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()
            out.append(pd.DataFrame(p, columns=LABELS).assign(StudyInstanceUID=list(sid)))
    return pd.concat(out)

for p in PLANES:
    auc, _ = evaluate(model, DataLoader(KneeDataset(val_by_plane[p]), batch_size=32, shuffle=False))
    print(f'val AUC on {p:9s}: {auc:.4f}')

allp = pd.concat([predict(val_by_plane[p]) for p in PLANES])
avg = allp.groupby('StudyInstanceUID')[LABELS].mean()
truth = (pd.concat(val_by_plane.values()).drop_duplicates('StudyInstanceUID')
         .set_index('StudyInstanceUID').loc[avg.index, LABELS] >= 0.5)
aucs = {l: roc_auc_score(truth[l], avg[l]) for l in LABELS if truth[l].nunique() == 2}
print(f'\nknee-level AUC (avg of 3 planes): {np.mean(list(aucs.values())):.4f}')
print(pd.Series(aucs).round(3).sort_values())

In [ ]:
# Cell 15: test-time slice pooling with the 09 model (no retraining)
WEIGHTS = '/kaggle/input/notebooks/alextanasescu/09-train-10ep/resnet18_all_planes_10ep.pt'
model = make_model()
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model.eval()

def slice_probs(df, slices):
    out = []
    with torch.no_grad():
        for path in df['path']:
            vol = np.load(path)[slices].astype(np.float32) / 255.0
            x = torch.from_numpy((vol - 0.45) / 0.225).unsqueeze(1).to(DEVICE)   # (k, 1, 224, 224)
            out.append(torch.sigmoid(model(x)).cpu().numpy())                     # (k, 12)
    return np.stack(out)                                                          # (knees, k, 12)

truth = (pd.concat(val_by_plane.values()).drop_duplicates('StudyInstanceUID')
         .set_index('StudyInstanceUID')[LABELS] >= 0.5)

def knee_auc(slices, pool):
    per_plane = []
    for p in PLANES:
        df = val_by_plane[p]
        probs = slice_probs(df, slices)
        agg = probs.max(1) if pool == 'max' else probs.mean(1)
        per_plane.append(pd.DataFrame(agg, columns=LABELS).assign(StudyInstanceUID=df['StudyInstanceUID'].values))
    avg = pd.concat(per_plane).groupby('StudyInstanceUID')[LABELS].mean()
    t = truth.loc[avg.index]
    aucs = {l: roc_auc_score(t[l], avg[l]) for l in LABELS if t[l].nunique() == 2}
    return np.mean(list(aucs.values())), pd.Series(aucs)

for name, slices, pool in [('middle only (sanity check)', [12], 'mean'),
                           ('slices 6-18, mean', list(range(6, 19, 2)), 'mean'),
                           ('slices 6-18, max', list(range(6, 19, 2)), 'max'),
                           ('all 24, mean', list(range(24)), 'mean')]:
    score, per = knee_auc(slices, pool)
    print(f'{name:28s} knee-level AUC {score:.4f}')